In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/NGC-3049_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3049/FORS2.2022-01-16T07_25_38.100/NGC-3049_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     387   (2094, 965)   float32   
  1  IMAGE.ERR     1 ImageHDU        68   (2094, 965)   float32   


In [3]:
image_cut = image[70:170, 0:1890]
image_error_cut = image_error[70:170, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00328
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) #+
       # sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/PSF/Real_seeing_NGC3773.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, a, b):
    func = model_convolved(x, params1, params2, params3)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    sum_area = blue_area + green_area +purple_area 
    total_area = total_integral(x, params1, params2, params3, a, b)
    print(blue_area)
    print(green_area)
    print(purple_area)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

30.170823153214457
0.7716066438654483
1.620467425853875
32.56289722293378
30.57260261051203
5.027833940355104
30.222619458993698
5.218392007667383
40.46884540701618
40.58032566461297
1.8002190934706053
0.006286239762330442
57.207813268043765
59.0143186012767
59.252203081999696
           Peak 1     Peak 2     Peak 3
Blue    92.653989  12.423962   3.050478
Green    2.369588  74.681200   0.010652
Purple   4.976423  12.894838  96.938869


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

54.88515600113314
2.7072297774490672
3.3748633876082943
60.9672491661905
58.212070260399884
15.188565174938228
85.44023409045565
15.990483746415709
116.61928301180959
116.87792472485813
5.416875511123443
0.033171641104548386
164.73005553095467
170.18010268318267
170.74570335487184
           Peak 1     Peak 2     Peak 3
Blue    90.023999  13.024060   3.183025
Green    4.440466  73.264242   0.019492
Purple   5.535535  13.711698  96.797483


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

74.51742677826414
7.701611725950026
5.408386559398144
87.6274250636123
84.61451413211921
25.805570015808005
122.59573887790486
28.143190153904794
176.54449904761765
176.76089674991448
9.10143846798093
0.1565085511533512
249.91825086199827
259.17619788113257
259.66082917516053
           Peak 1     Peak 2     Peak 3
Blue    85.038932  14.617034   3.511680
Green    8.789043  69.441835   0.060387
Purple   6.172025  15.941131  96.427933


## HBeta


In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

11.25232714348508
8.52415698227146
0.6610406624919607
20.4375247882485
20.39659195829255
1.3836351013055648
10.244127270667
2.5368386183763585
14.164600990348923
14.192691754175847
0.5798135884510354
2.5348337209800214
18.057805143043893
21.17245245247495
21.32744170393488
           Peak 1     Peak 2     Peak 3
Blue    55.057191   9.768260   2.738528
Green   41.708363  72.322032  11.972320
Purple   3.234446  17.909708  85.289152


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

21.439881618325145
17.288289629230917
1.389489112638204
40.117660360194265
40.18579378771173
4.179783026768657
30.072414472194062
7.786657473153415
42.03885497211613
42.10086241914924
1.743261619268975
7.665268077421588
52.80839477588785
62.21692447257841
62.65742837937338
           Peak 1     Peak 2     Peak 3
Blue    53.442502   9.942666   2.801909
Green   43.093963  71.534809  12.320230
Purple   3.463535  18.522525  84.877861


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


29.386871704745253
26.530117665788797
2.2590270670690487
58.1760164376031
58.33080381830732
7.102147140689313
47.54362558993088
13.759358632593699
68.4051313632139
68.44959147334872
2.9227285251717916
13.049030576803778
82.03646485523981
98.00822395721538
98.57429903743892
           Peak 1     Peak 2     Peak 3
Blue    50.513723  10.382477   2.982126
Green   45.603187  69.503010  13.314220
Purple   3.883090  20.114512  83.703654


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] - sigma, 
                                                                fit_HA['Component 1'].iloc[3] +  sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] - sigma, 
                                                                fit_HA['Component 2'].iloc[3] +  sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] - sigma, 
                                                                fit_HA['Component 3'].iloc[3] +  sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

12.178383963341233
3.1026583598435353
0.11789646813311419
15.398938791317882
15.097892656853768
1.4787645728676513
5.388469235668463
1.0471029960625404
7.914336804598655
8.214488432575568
0.3489882624390326
1.0833744404059584
8.603240930454696
10.035603633299687
10.051230921804143
           Peak 1     Peak 2     Peak 3
Blue    79.085865  18.684630   3.477501
Green   20.148521  68.084912  10.795309
Purple   0.765614  13.230458  85.727189


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

21.979797806129053
6.319337072207834
0.2640473080474505
28.56318218638434
28.155860701053598
4.488213306764187
15.386344200738424
3.2372020235458394
23.111759531048452
23.84953738482485
1.0528996758856513
3.266149997520893
25.23761334373211
29.556663017138654
29.596860698302226
           Peak 1     Peak 2     Peak 3
Blue    76.951502  19.419609   3.562309
Green   22.124065  66.573660  11.050469
Purple   0.924432  14.006731  85.387222


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

29.401979617393387
9.796190100471293
0.4712271174678147
39.66939683533249
39.22167424798074
7.724320590155417
22.796609356726453
5.815341458504109
36.33627140538598
37.11857994473912
1.7817909913980121
5.51774380787918
39.74930618832695
47.04884098760414
47.09858832318177
           Peak 1     Peak 2     Peak 3
Blue    74.117536  21.257879   3.787109
Green   24.694578  62.737888  11.727693
Purple   1.187886  16.004233  84.485197


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/Fit/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

10.296895026858744
2.5432311287577134
0.16378807826487127
13.003914233881327
13.008727210837836
1.3832155871280927
3.4097794448701175
1.0748824014741627
5.867877433472373
5.764146840336096
0.3708341275704996
0.8481693034901172
4.979755124690907
6.198758555751525
6.202937499662868
           Peak 1     Peak 2     Peak 3
Blue    79.183043  23.572673   5.982393
Green   19.557428  58.109248  13.682890
Purple   1.259529  18.318079  80.334717


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

18.801876991066152
5.164229327828188
0.3609231037114233
24.327029422605765
24.34062090918385
4.191529318559004
9.8381650670628
3.2832638255130275
17.31295821113483
17.04624758335832
1.1177973630090572
2.553783624588842
14.73512825404896
18.40670924164686
18.418850620695935
          Peak 1     Peak 2     Peak 3
Blue    77.28801  24.210359   6.072771
Green   21.22836  56.825442  13.874200
Purple   1.48363  18.964199  80.053029


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

25.404154624893486
7.962113845625026
0.6280722473036905
33.994340717822205
33.99548734984871
7.182141913991583
14.855245244358333
5.727046854997999
27.764434013347916
27.470725499781175
1.8870089819052314
4.298904069312314
23.696791612687225
29.88270466390477
29.901952956352307
           Peak 1     Peak 2     Peak 3
Blue    74.730541  25.868137   6.314720
Green   23.421880  53.504585  14.385927
Purple   1.847579  20.627278  79.299354


## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3773/Fit/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

21.1579338487052
5.383674863911311
0.5122905552868745
27.053899267903383
27.135989629576763
3.0956040435415346
17.86363607225987
3.7871673996508592
24.746407515452262
30.177107002280856
0.5290795076922223
1.826979713900735
32.13122844290286
34.48728766449582
34.67958720662063
           Peak 1     Peak 2     Peak 3
Blue    78.206597  12.509307   1.534129
Green   19.899811  72.186785   5.297545
Purple   1.893592  15.303908  93.168326


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

39.654984845721046
11.088267762758079
1.1216472896222203
51.86489989810134
51.97304715466396
9.387989825419197
49.80499756196962
11.764477548726319
70.95746493611514
84.57445230727626
1.6019444205953388
5.525131866578178
94.48234082665793
101.60941711383146
102.11943287661649
           Peak 1     Peak 2     Peak 3
Blue    76.458231  13.230447   1.576571
Green   21.379137  70.189934   5.437618
Purple   2.162633  16.579619  92.985811


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

54.810356534579384
17.557864911778758
1.9404216925675921
74.30864313892575
74.38813698599998
16.109831528400324
70.31142798344848
21.39286269853113
107.81412221037994
122.79005010891245
2.736744439568979
9.416139129107759
148.24712609756273
160.40000966623947
160.95879721245217
           Peak 1     Peak 2     Peak 3
Blue    73.760406  14.942228   1.706200
Green   23.628294  65.215416   5.870411
Purple   2.611300  19.842357  92.423390
